# Modul 20: Fairer Modellvergleich und verantwortungsvolles Abschlussprojekt

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Fair vergleichen, Projekt umsetzen  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittenes Integrations- und Entscheidungsprojekt  
    **Orientierungszeit:** etwa 180 bis 240 Minuten

    ## Überblick

    Sie planen und dokumentieren ein vollständiges Klassifikationsprojekt. Identische Splits, Baselines, Leistungs- und Ressourcenmetriken, reproduzierbare Artefakte, sichere Inferenz, Teilgruppenprüfungen, Verteilungsverschiebung, Fehleranalyse und eine Modellkarte werden zu einem nachvollziehbaren Abschlussworkflow verbunden.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_20A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_20B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Problem, Ziel, Datenherkunft, Spaltenbedeutung und mögliche Leakage vor der Modellierung dokumentieren.
- EDA, Split, Baseline und mehrere Modelle unter identischen Bedingungen verbinden.
- Leistung, Trainingszeit, Inferenzzeit, Komplexität und Artefaktgröße fair vergleichen.
- Modell, Vorverarbeitung, Metadaten und Referenztests reproduzierbar speichern und laden.
- Eingaben vor der Inferenz validieren und Fehler verständlich behandeln.
- Teilgruppenleistung und einfache Verteilungsverschiebungen untersuchen.
- Verbesserungen nur mit Trainings- und Validierungsdaten auswählen.
- Eine kompakte Modellkarte und einen technisch begründeten Projektbericht erstellen.

    ## Bewertete Fähigkeiten

    - Projektplanung, Datensteckbrief, EDA und Leakage-Prüfung
- faire Splits, Baselines, Modellvergleich und Ressourcenmessung
- Versionierung, Speichern, Laden und sichere Inferenz
- Teilgruppenmetriken, Verteilungsverschiebung und Modellkarte
- Fehleranalyse, Schwellenwertwahl und Abschlussbericht

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import io
import json
import pickle
import platform
import time

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

cancer_20 = load_breast_cancer(as_frame=True)
project_data_20 = cancer_20.data.copy()

# Positive Klasse 1 bedeutet in diesem Projekt "maligne". Diese explizite
# Umcodierung macht Recall und Fehlertypen fachlich leichter lesbar.
project_data_20["malignant"] = (cancer_20.target == 0).astype(int)
project_data_20["record_id"] = np.arange(100000, 100000 + len(project_data_20))

# Eine synthetische Standortgruppe dient ausschließlich der technischen
# Teilgruppenanalyse. Sie ist kein geschütztes personenbezogenes Merkmal.
texture_median_20 = float(project_data_20["mean texture"].median())
project_data_20["acquisition_site"] = np.where(
    project_data_20["mean texture"] <= texture_median_20,
    "Site_A",
    "Site_B",
)

# Diese absichtlich ungeeignete Spalte simuliert eine Information, die
# erst nach der endgültigen Diagnose vorliegt und deshalb Leakage wäre.
project_data_20["diagnosis_after_review"] = project_data_20["malignant"]

feature_columns_20 = [str(name) for name in cancer_20.feature_names]
target_column_20 = "malignant"
group_column_20 = "acquisition_site"
leakage_columns_20 = ["diagnosis_after_review"]
identifier_columns_20 = ["record_id"]

print("Projektdataframe:", project_data_20.shape)
print("Positive Klasse 1 = maligne")

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Projekt planen, Daten beschreiben und Leakage erkennen

    Erstellen Sie den Daten- und Problemsteckbrief für das Abschlussprojekt.

1. Formulieren Sie Problem, Zielvariable, positive Klasse, Analyseeinheit und primäre Fehlerrisiken.
2. Erstellen Sie eine Tabelle mit Spaltenname, Datentyp, Rolle und kurzer Bedeutung für mindestens zehn Merkmale sowie Ziel, Gruppe, ID und die verdächtige Nachdiagnose-Spalte.
3. Markieren Sie Identifier und potenzielle Leakage-Spalten und begründen Sie deren Ausschluss aus den Modellmerkmalen.
4. Prüfen Sie Form, Duplikate, Fehlwerte, Klassenverteilung und Gruppenverteilung.
5. Visualisieren Sie zwei fachlich sinnvolle Merkmale nach Zielklasse und eine Korrelationsübersicht für eine kleine Merkmalsauswahl.
6. Formulieren Sie drei vorsichtige EDA-Befunde, ohne Kausalität zu behaupten.

> **Hinweis:** Fragen Sie für jede Spalte, ob ihr Wert zum realen Vorhersagezeitpunkt bereits verfügbar wäre.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Projekt planen, Daten beschreiben und Leakage erkennen
#
# Ziel dieser Codezelle:
# Erstellen Sie den Daten- und Problemsteckbrief für das Abschlussprojekt. 1.
# Formulieren Sie Problem, Zielvariable, positive Klasse, Analyseeinheit und primäre
# Fehlerrisiken. 2. Erstellen Sie eine Tabelle mit Spaltenna...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

project_definition_20 = {
    "problem": "Binäre Klassifikation maligner gegenüber benigner Befunde aus numerischen Messmerkmalen.",
    "unit_of_analysis": "ein untersuchter Befund beziehungsweise Datensatzzeile",
    "target": target_column_20,
    "positive_class": "1 = maligne",
    "primary_error_risk": "Ein falsch-negatives Ergebnis kann einen malignen Befund übersehen.",
    "secondary_error_risk": "Ein falsch-positives Ergebnis kann unnötige weitere Untersuchungen auslösen.",
    "intended_use": "Lehrdemonstration eines reproduzierbaren ML-Workflows, nicht klinische Diagnose.",
}
print("Projektdefinition:")
for key, value in project_definition_20.items():
    print(f"- {key}: {value}")

# Ein kompakter Datensteckbrief macht Rollen vor der Modellierung klar.
described_features_20 = feature_columns_20[:10]
description_rows_20 = []
for column in described_features_20:
    description_rows_20.append(
        {
            "column": column,
            "dtype": str(project_data_20[column].dtype),
            "role": "numerisches Merkmal",
            "meaning": "Messgröße aus dem eingebauten Breast-Cancer-Datensatz",
        }
    )
description_rows_20.extend(
    [
        {
            "column": target_column_20,
            "dtype": str(project_data_20[target_column_20].dtype),
            "role": "Ziel",
            "meaning": "1 maligne, 0 benigne",
        },
        {
            "column": group_column_20,
            "dtype": str(project_data_20[group_column_20].dtype),
            "role": "Analysegruppe",
            "meaning": "synthetischer Messstandort für technische Teilgruppenprüfung",
        },
        {
            "column": "record_id",
            "dtype": str(project_data_20["record_id"].dtype),
            "role": "Identifier, ausschließen",
            "meaning": "eindeutige Zeilenkennung ohne fachliches Vorhersagesignal",
        },
        {
            "column": "diagnosis_after_review",
            "dtype": str(project_data_20["diagnosis_after_review"].dtype),
            "role": "Leakage, ausschließen",
            "meaning": "Information liegt erst nach der zu prognostizierenden Diagnose vor",
        },
    ]
)
data_dictionary_20 = pd.DataFrame(description_rows_20)
print("\nDatensteckbrief:")
print(data_dictionary_20.to_string(index=False))

quality_summary_20 = pd.Series(
    {
        "rows": len(project_data_20),
        "columns": project_data_20.shape[1],
        "duplicate_rows": int(project_data_20.duplicated().sum()),
        "missing_values": int(project_data_20.isna().sum().sum()),
        "positive_class_rate": float(project_data_20[target_column_20].mean()),
    }
)
print("\nQualitätsübersicht:")
print(quality_summary_20.round(4))
print("\nKlassenverteilung:")
print(project_data_20[target_column_20].value_counts(normalize=True).sort_index().round(3))
print("\nGruppenverteilung:")
print(project_data_20[group_column_20].value_counts(normalize=True).round(3))

# Boxplots zeigen Verteilungen und mögliche Gruppenunterschiede, aber
# sie belegen keine Ursache-Wirkungs-Beziehung.
for feature in ["mean radius", "mean concavity"]:
    fig, ax = plt.subplots(figsize=(7, 4))
    project_data_20.boxplot(column=feature, by=target_column_20, ax=ax)
    ax.set_title(f"{feature} nach Zielklasse")
    ax.set_xlabel("maligne Klasse")
    ax.set_ylabel(feature)
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

correlation_columns_20 = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean concavity",
    target_column_20,
]
correlation_matrix_20 = project_data_20[correlation_columns_20].corr()
fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(correlation_matrix_20, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(correlation_columns_20)), correlation_columns_20, rotation=70, ha="right")
ax.set_yticks(range(len(correlation_columns_20)), correlation_columns_20)
ax.set_title("Korrelationsübersicht ausgewählter Variablen")
fig.colorbar(image, ax=ax, label="Korrelation")
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 1

Mögliche vorsichtige Befunde sind: Mehrere Größenmerkmale unterscheiden sich in diesem Datensatz sichtbar zwischen den Zielklassen; einige Messgrößen sind stark miteinander korreliert und enthalten daher teilweise redundante Information; die positive Klasse ist nicht exakt ausgeglichen, weshalb Balanced Accuracy und Recall zusätzlich zur Accuracy sinnvoll sind. Keine dieser Beobachtungen beweist Kausalität oder klinische Einsatzfähigkeit. `diagnosis_after_review` wäre perfekte, aber unzulässige Zielinformation. `record_id` ist eine Kennung und kein belastbares fachliches Merkmal.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Baseline und Modelle unter identischen Bedingungen vergleichen

    Bauen Sie einen fairen Modellvergleich.

1. Trennen Sie zunächst 20 Prozent als Testset und danach 25 Prozent des verbleibenden Teils als Validierung. Verwenden Sie Stratifikation und feste Seeds.
2. Verwenden Sie ausschließlich `feature_columns_20` als Modellmerkmale.
3. Vergleichen Sie eine häufigste-Klasse-Baseline, logistische Regression mit Skalierung, einen Entscheidungsbaum und einen kleinen Random Forest.
4. Trainieren Sie alle Modelle auf demselben Training und bewerten Sie sie auf derselben Validierung.
5. Messen Sie Accuracy, Balanced Accuracy, Recall der malignen Klasse, ROC-AUC, Trainingszeit, Inferenzzeit pro Validierungsbatch und serialisierte Größe.
6. Wählen Sie das Modell anhand einer vorher festgelegten Regel: höchste Validierungs-Balanced-Accuracy, bei Gleichstand höherer maligner Recall, danach kleinere Artefaktgröße.
7. Bewerten Sie das ausgewählte Modell noch nicht auf dem Testset.

> **Hinweis:** Definieren Sie die Auswahlregel vor dem Blick auf das Testset.

In [ ]:
# Speichern Sie das ausgewählte Modell als selected_model_20 und den
# Namen als selected_model_name_20.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Baseline und Modelle unter identischen Bedingungen vergleichen
#
# Ziel dieser Codezelle:
# Bauen Sie einen fairen Modellvergleich. 1. Trennen Sie zunächst 20 Prozent als
# Testset und danach 25 Prozent des verbleibenden Teils als Validierung. Verwenden
# Sie Stratifikation und feste Seeds. 2. Verwenden Sie auss...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

X_all_20 = project_data_20[feature_columns_20].copy()
y_all_20 = project_data_20[target_column_20].to_numpy()
groups_all_20 = project_data_20[group_column_20].to_numpy()

X_train_valid_20, X_test_20, y_train_valid_20, y_test_20, groups_train_valid_20, groups_test_20 = train_test_split(
    X_all_20,
    y_all_20,
    groups_all_20,
    test_size=0.20,
    stratify=y_all_20,
    random_state=RANDOM_SEED,
)
X_train_20, X_valid_20, y_train_20, y_valid_20, groups_train_20, groups_valid_20 = train_test_split(
    X_train_valid_20,
    y_train_valid_20,
    groups_train_valid_20,
    test_size=0.25,
    stratify=y_train_valid_20,
    random_state=RANDOM_SEED,
)

models_20 = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "LogisticRegression": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    C=1.0,
                    max_iter=3000,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    ),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=8,
        random_state=RANDOM_SEED,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=120,
        max_depth=6,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=RANDOM_SEED,
    ),
}

validation_rows_20 = []
fitted_models_20 = {}
for model_name, model in models_20.items():
    fit_start_20 = time.perf_counter()
    model.fit(X_train_20, y_train_20)
    fit_seconds_20 = time.perf_counter() - fit_start_20

    # Ein Aufwärmlauf reduziert einmalige Initialisierungseinflüsse.
    _ = model.predict(X_valid_20.iloc[:16])
    inference_durations_20 = []
    for _ in range(5):
        inference_start_20 = time.perf_counter()
        _ = model.predict(X_valid_20)
        inference_durations_20.append(time.perf_counter() - inference_start_20)

    valid_predictions_20 = model.predict(X_valid_20)
    if hasattr(model, "predict_proba"):
        valid_probabilities_20 = model.predict_proba(X_valid_20)[:, 1]
        valid_auc_20 = roc_auc_score(y_valid_20, valid_probabilities_20)
    else:
        valid_probabilities_20 = valid_predictions_20.astype(float)
        valid_auc_20 = np.nan

    serialized_model_20 = pickle.dumps(model)
    validation_rows_20.append(
        {
            "model": model_name,
            "validation_accuracy": accuracy_score(y_valid_20, valid_predictions_20),
            "validation_balanced_accuracy": balanced_accuracy_score(y_valid_20, valid_predictions_20),
            "validation_malignant_recall": recall_score(y_valid_20, valid_predictions_20, pos_label=1),
            "validation_roc_auc": valid_auc_20,
            "fit_seconds": fit_seconds_20,
            "median_inference_seconds": float(np.median(inference_durations_20)),
            "artifact_kilobytes": len(serialized_model_20) / 1000.0,
        }
    )
    fitted_models_20[model_name] = model

model_comparison_20 = pd.DataFrame(validation_rows_20).sort_values(
    [
        "validation_balanced_accuracy",
        "validation_malignant_recall",
        "artifact_kilobytes",
    ],
    ascending=[False, False, True],
)
selected_model_name_20 = str(model_comparison_20.iloc[0]["model"])
selected_model_20 = fitted_models_20[selected_model_name_20]

assert list(X_train_20.columns) == feature_columns_20
print("Train/Valid/Test:", X_train_20.shape, X_valid_20.shape, X_test_20.shape)
print(model_comparison_20.round(5).to_string(index=False))
print("\nAuswahl nach vorab definierter Regel:", selected_model_name_20)

### Reflexion zu Aufgabe 2

Identische Datenpartitionen, Metriken und Messverfahren sind die Grundlage eines fairen Vergleichs. Accuracy kann bei ungleichen Klassen irreführend sein. Maligner Recall misst hier den Anteil erkannter positiver Fälle, während Balanced Accuracy beide Klassen gleich gewichtet. ROC-AUC bewertet die Rangordnung über viele Schwellenwerte. Laufzeit- und Größenwerte hängen von Hardware und Softwareumgebung ab und sind deshalb als relative Messungen dieses Laufs zu dokumentieren.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Modellversion speichern, laden und Inferenz absichern

    Erstellen Sie ein reproduzierbares Modellartefakt für `selected_model_20`.

1. Speichern Sie Modell, erwartete Spaltenreihenfolge, Zielcodierung, Gruppenspalte, Seed, Schwellenwert, Paketversionen und Trainingszeitstempel in einem Dictionary.
2. Serialisieren Sie das Artefakt in `io.BytesIO` und laden Sie es neu.
3. Schreiben Sie eine Funktion `safe_predict_20`, die einen DataFrame erwartet und fehlende, zusätzliche oder falsch sortierte Spalten, Fehlwerte, unendliche Werte und leere Eingaben verständlich ablehnt.
4. Geben Sie Wahrscheinlichkeit und Klasse zurück.
5. Prüfen Sie zwölf Referenzzeilen vor und nach dem Laden auf identische Ergebnisse.
6. Demonstrieren Sie mindestens zwei kontrolliert abgefangene ungültige Eingaben.

> **Hinweis:** Validieren Sie das Schema vor jeder Modellmethode, nicht erst nach einem Fehler.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Modellversion speichern, laden und Inferenz absichern
#
# Ziel dieser Codezelle:
# Erstellen Sie ein reproduzierbares Modellartefakt für selectedmodel20. 1.
# Speichern Sie Modell, erwartete Spaltenreihenfolge, Zielcodierung, Gruppenspalte,
# Seed, Schwellenwert, Paketversionen und Trainingszeitstempel...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

training_timestamp_20 = pd.Timestamp.utcnow().isoformat()
model_artifact_20 = {
    "model": selected_model_20,
    "metadata": {
        "model_name": selected_model_name_20,
        "feature_columns": feature_columns_20,
        "target": target_column_20,
        "positive_class": "1 = maligne",
        "group_column": group_column_20,
        "random_seed": RANDOM_SEED,
        "decision_threshold": 0.5,
        "trained_at_utc": training_timestamp_20,
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "sklearn_version": sklearn.__version__,
    },
}

artifact_buffer_20 = io.BytesIO()
pickle.dump(model_artifact_20, artifact_buffer_20)
artifact_size_20 = artifact_buffer_20.getbuffer().nbytes
artifact_buffer_20.seek(0)
loaded_artifact_20 = pickle.load(artifact_buffer_20)

def safe_predict_20(artifact, input_frame):
    if not isinstance(input_frame, pd.DataFrame):
        raise TypeError("Die Eingabe muss ein pandas DataFrame sein.")
    if input_frame.empty:
        raise ValueError("Die Eingabe enthält keine Zeilen.")

    expected_columns = artifact["metadata"]["feature_columns"]
    actual_columns = list(input_frame.columns)
    missing_columns = [column for column in expected_columns if column not in actual_columns]
    extra_columns = [column for column in actual_columns if column not in expected_columns]
    if missing_columns or extra_columns:
        raise ValueError(
            f"Spalten stimmen nicht. Fehlend: {missing_columns}; zusätzlich: {extra_columns}"
        )
    if actual_columns != expected_columns:
        raise ValueError("Die Spaltenreihenfolge stimmt nicht mit dem Artefakt überein.")
    if input_frame.isna().any().any():
        raise ValueError("Die Eingabe enthält Fehlwerte.")

    numeric_values = input_frame.to_numpy(dtype=float)
    if not np.isfinite(numeric_values).all():
        raise ValueError("Die Eingabe enthält unendliche oder ungültige Zahlen.")

    model = artifact["model"]
    threshold = artifact["metadata"]["decision_threshold"]
    probabilities = model.predict_proba(input_frame)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    return pd.DataFrame(
        {
            "malignant_probability": probabilities,
            "predicted_class": predictions,
        },
        index=input_frame.index,
    )

reference_input_20 = X_valid_20.iloc[:12].copy()
original_reference_20 = safe_predict_20(model_artifact_20, reference_input_20)
loaded_reference_20 = safe_predict_20(loaded_artifact_20, reference_input_20)
pd.testing.assert_frame_equal(original_reference_20, loaded_reference_20)

print("Artefaktgröße in Bytes:", artifact_size_20)
print("Metadaten:", loaded_artifact_20["metadata"])
print("Referenzprüfung bestanden.")

invalid_examples_20 = {
    "fehlende Spalte": reference_input_20.drop(columns=[feature_columns_20[0]]),
    "Fehlwert": reference_input_20.assign(**{feature_columns_20[1]: np.nan}),
    "falsche Reihenfolge": reference_input_20[feature_columns_20[::-1]],
}
for description, invalid_frame in invalid_examples_20.items():
    try:
        safe_predict_20(loaded_artifact_20, invalid_frame)
    except (TypeError, ValueError) as error:
        print(f"Kontrollierter Fehler ({description}): {error}")

### Reflexion zu Aufgabe 3

Ein reproduzierbares Artefakt umfasst mehr als das trainierte Objekt. Spaltenschema, Vorverarbeitung, Zielbedeutung, Schwelle, Versionen und Referenztests verhindern viele stille Fehler. `pickle` sollte nur aus vertrauenswürdigen Quellen geladen werden, weil das Format beim Laden Code ausführen kann. Für echte Bereitstellung sind Zugriffskontrolle, signierte Artefakte, Versionsverwaltung und sichere Serialisierungsverfahren zu prüfen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Teilgruppen, Verteilungsverschiebung und Modellkarte prüfen

    Bewerten Sie das ausgewählte Modell verantwortungsvoll, zunächst auf der Validierung.

1. Schreiben Sie eine Funktion, die pro `acquisition_site` Beispielzahl, positive Rate, Accuracy, Balanced Accuracy, malignen Recall, Präzision und Spezifität berechnet.
2. Wenden Sie sie auf die Validierungsdaten mit der Standardschwelle 0.5 an.
3. Berichten Sie die größte absolute Lücke zwischen den Gruppen für malignen Recall und Spezifität.
4. Simulieren Sie eine einfache Messverschiebung, indem Sie für `Site_B` in einer Kopie der Validierungsmerkmale drei ausgewählte Spalten um 1.50 Trainingsstandardabweichungen verringern.
5. Vergleichen Sie Gesamt- und Gruppenmetriken vor und nach der Verschiebung.
6. Erstellen Sie eine strukturierte Modellkarte mit Zweck, Daten, Metriken, Gruppenprüfung, Grenzen, Risiken und nicht vorgesehenen Anwendungen.

> **Hinweis:** Berechnen Sie Spezifität aus TN und FP und verwenden Sie für jede Gruppe dieselbe Schwelle.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Teilgruppen, Verteilungsverschiebung und Modellkarte prüfen
#
# Ziel dieser Codezelle:
# Bewerten Sie das ausgewählte Modell verantwortungsvoll, zunächst auf der
# Validierung. 1. Schreiben Sie eine Funktion, die pro acquisitionsite Beispielzahl,
# positive Rate, Accuracy, Balanced Accuracy, malignen Recall,...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

def subgroup_metrics_20(y_true, probabilities, groups, threshold=0.5):
    rows = []
    predictions = (np.asarray(probabilities) >= threshold).astype(int)
    y_true = np.asarray(y_true).astype(int)
    groups = np.asarray(groups)

    for group_name in np.unique(groups):
        mask = groups == group_name
        group_y = y_true[mask]
        group_predictions = predictions[mask]
        matrix = confusion_matrix(group_y, group_predictions, labels=[0, 1])
        true_negative, false_positive, false_negative, true_positive = matrix.ravel()
        specificity = true_negative / max(true_negative + false_positive, 1)
        rows.append(
            {
                "group": group_name,
                "n": int(mask.sum()),
                "positive_rate": float(group_y.mean()),
                "accuracy": accuracy_score(group_y, group_predictions),
                "balanced_accuracy": balanced_accuracy_score(group_y, group_predictions),
                "malignant_recall": recall_score(group_y, group_predictions, pos_label=1, zero_division=0),
                "precision": precision_score(group_y, group_predictions, pos_label=1, zero_division=0),
                "specificity": specificity,
            }
        )
    return pd.DataFrame(rows)

valid_probabilities_20 = selected_model_20.predict_proba(X_valid_20)[:, 1]
subgroup_before_20 = subgroup_metrics_20(
    y_valid_20,
    valid_probabilities_20,
    groups_valid_20,
    threshold=0.5,
)
recall_gap_20 = float(
    subgroup_before_20["malignant_recall"].max()
    - subgroup_before_20["malignant_recall"].min()
)
specificity_gap_20 = float(
    subgroup_before_20["specificity"].max()
    - subgroup_before_20["specificity"].min()
)

print("Teilgruppen vor Verschiebung:")
print(subgroup_before_20.round(4).to_string(index=False))
print("Recall-Lücke:", round(recall_gap_20, 4))
print("Spezifitäts-Lücke:", round(specificity_gap_20, 4))

shifted_valid_20 = X_valid_20.copy()
shifted_columns_20 = ["mean radius", "worst radius", "mean concavity"]
site_b_mask_20 = groups_valid_20 == "Site_B"
training_scales_20 = X_train_20[shifted_columns_20].std(ddof=0)
shifted_valid_20.loc[site_b_mask_20, shifted_columns_20] -= 1.50 * training_scales_20

shifted_probabilities_20 = selected_model_20.predict_proba(shifted_valid_20)[:, 1]
shifted_predictions_20 = (shifted_probabilities_20 >= 0.5).astype(int)
subgroup_after_20 = subgroup_metrics_20(
    y_valid_20,
    shifted_probabilities_20,
    groups_valid_20,
    threshold=0.5,
)

shift_summary_20 = pd.DataFrame(
    {
        "condition": ["original", "shifted Site_B"],
        "balanced_accuracy": [
            balanced_accuracy_score(y_valid_20, valid_probabilities_20 >= 0.5),
            balanced_accuracy_score(y_valid_20, shifted_predictions_20),
        ],
        "malignant_recall": [
            recall_score(y_valid_20, valid_probabilities_20 >= 0.5),
            recall_score(y_valid_20, shifted_predictions_20),
        ],
    }
)
print("\nGesamtvergleich der Verschiebung:")
print(shift_summary_20.round(4).to_string(index=False))
print("\nTeilgruppen nach Verschiebung:")
print(subgroup_after_20.round(4).to_string(index=False))

model_card_20 = {
    "model_name": selected_model_name_20,
    "intended_use": project_definition_20["intended_use"],
    "not_intended_for": "Keine autonome klinische Diagnose oder Therapieentscheidung.",
    "training_data": "scikit-learn Breast Cancer Wisconsin Diagnostic, fester Train/Valid/Test-Split",
    "target": "1 = maligne, 0 = benigne",
    "selection_metric": "Validierungs-Balanced-Accuracy mit Recall- und Größen-Tiebreak",
    "validation_subgroup_check": subgroup_before_20.round(4).to_dict(orient="records"),
    "shift_test": shift_summary_20.round(4).to_dict(orient="records"),
    "known_limitations": [
        "kleiner Lehrdatensatz",
        "synthetische Standortgruppe",
        "keine externe Validierung",
        "keine Kalibrierungsstudie",
        "Ressourcenmessungen gelten nur für diese Umgebung",
    ],
    "risks": [
        "falsch-negative maligne Fälle",
        "Leistungsänderung bei Datenverschiebung",
        "unbekannte Leistung in anderen Populationen und Geräten",
    ],
}
print("\nModellkarte:")
print(json.dumps(model_card_20, ensure_ascii=False, indent=2))

### Reflexion zu Aufgabe 4

Teilgruppenmetriken benötigen ausreichende Gruppengrößen und fachlich sinnvolle Gruppen. Unterschiede können durch Grundraten, Messbedingungen, Stichprobenzufall oder echte Modellschwächen entstehen und beweisen nicht allein Diskriminierung. Der simulierte Shift ist ein Sensitivitätstest, keine realistische externe Validierung. Eine Modellkarte dokumentiert bekannte Fakten und Grenzen, ersetzt aber keine unabhängige Prüfung, Überwachung oder menschliche Verantwortung.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integration: Fehler analysieren, Schwelle verbessern und Projektbericht abschließen

    Schließen Sie das Projekt ab, ohne das Testset zur Optimierung zu verwenden.

1. Analysieren Sie falsch-negative Validierungsfälle des ausgewählten Modells und vergleichen Sie deren ausgewählte Merkmale mit korrekt erkannten malignen Fällen.
2. Untersuchen Sie Schwellenwerte von 0.10 bis 0.90 auf der Validierung.
3. Wählen Sie den höchsten Schwellenwert, der mindestens 95 Prozent malignen Recall auf der Validierung erreicht. Falls keiner dies schafft, wählen Sie den Schwellenwert mit dem höchsten Recall und danach höchster Balanced Accuracy.
4. Vergleichen Sie auf der Validierung Standardschwelle und neue Schwelle anhand von Recall, Spezifität, Präzision und Balanced Accuracy.
5. Aktualisieren Sie die Artefaktmetadaten mit der gewählten Schwelle.
6. Bewerten Sie **erst jetzt** das unveränderte Testset mit der finalen Schwelle und berichten Sie Gesamt- und Teilgruppenmetriken.
7. Erstellen Sie einen kompakten Projektbericht mit Problem, Daten, Methode, Auswahlregel, Baseline, finalen Ergebnissen, Ressourcen, Fehleranalyse, Grenzen und nächsten Schritten.

> **Hinweis:** Verwenden Sie das Testset erst, nachdem Modell und Schwelle vollständig festgelegt sind.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integration: Fehler analysieren, Schwelle verbessern und Projektbericht abschließen
#
# Ziel dieser Codezelle:
# Schließen Sie das Projekt ab, ohne das Testset zur Optimierung zu verwenden. 1.
# Analysieren Sie falsch-negative Validierungsfälle des ausgewählten Modells und
# vergleichen Sie deren ausgewählte Merkmale mit korrekt erk...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

valid_predictions_default_20 = (valid_probabilities_20 >= 0.5).astype(int)
false_negative_mask_20 = (y_valid_20 == 1) & (valid_predictions_default_20 == 0)
true_positive_mask_20 = (y_valid_20 == 1) & (valid_predictions_default_20 == 1)
analysis_features_20 = ["mean radius", "mean texture", "mean concavity", "worst area"]

error_analysis_rows_20 = []
for feature in analysis_features_20:
    error_analysis_rows_20.append(
        {
            "feature": feature,
            "false_negative_mean": float(X_valid_20.loc[false_negative_mask_20, feature].mean()),
            "true_positive_mean": float(X_valid_20.loc[true_positive_mask_20, feature].mean()),
            "false_negative_count": int(false_negative_mask_20.sum()),
        }
    )
error_analysis_20 = pd.DataFrame(error_analysis_rows_20)
print("Validierungs-Fehleranalyse:")
print(error_analysis_20.round(4).to_string(index=False))

threshold_rows_20 = []
for threshold in np.linspace(0.10, 0.90, 81):
    predictions = (valid_probabilities_20 >= threshold).astype(int)
    matrix = confusion_matrix(y_valid_20, predictions, labels=[0, 1])
    true_negative, false_positive, false_negative, true_positive = matrix.ravel()
    specificity = true_negative / max(true_negative + false_positive, 1)
    threshold_rows_20.append(
        {
            "threshold": float(threshold),
            "malignant_recall": recall_score(y_valid_20, predictions, zero_division=0),
            "specificity": specificity,
            "precision": precision_score(y_valid_20, predictions, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_valid_20, predictions),
        }
    )
threshold_table_20 = pd.DataFrame(threshold_rows_20)

eligible_thresholds_20 = threshold_table_20[
    threshold_table_20["malignant_recall"] >= 0.95
]
if not eligible_thresholds_20.empty:
    # Der höchste noch ausreichend sensible Schwellenwert reduziert
    # tendenziell unnötige positive Meldungen unter der Recall-Vorgabe.
    selected_threshold_20 = float(eligible_thresholds_20["threshold"].max())
else:
    fallback_row_20 = threshold_table_20.sort_values(
        ["malignant_recall", "balanced_accuracy"],
        ascending=[False, False],
    ).iloc[0]
    selected_threshold_20 = float(fallback_row_20["threshold"])

default_validation_row_20 = threshold_table_20.iloc[
    (threshold_table_20["threshold"] - 0.5).abs().argmin()
]
selected_validation_row_20 = threshold_table_20.iloc[
    (threshold_table_20["threshold"] - selected_threshold_20).abs().argmin()
]
validation_threshold_comparison_20 = pd.DataFrame(
    [default_validation_row_20, selected_validation_row_20],
    index=["Standard 0.5", "validierungsbasiert"],
)
print("\nSchwellenvergleich auf Validierung:")
print(validation_threshold_comparison_20.round(4).to_string())

# Das finale Artefakt übernimmt die Entscheidung, die ausschließlich
# aus Trainings- und Validierungsinformationen hervorging.
final_artifact_20 = loaded_artifact_20
final_artifact_20["metadata"]["decision_threshold"] = selected_threshold_20
final_artifact_20["metadata"]["threshold_selection"] = (
    "höchster Validierungsschwellenwert mit mindestens 0.95 malignem Recall"
)

# Erst jetzt wird das Testset für die finale Berichterstattung geöffnet.
test_probabilities_20 = selected_model_20.predict_proba(X_test_20)[:, 1]
test_predictions_20 = (test_probabilities_20 >= selected_threshold_20).astype(int)
test_matrix_20 = confusion_matrix(y_test_20, test_predictions_20, labels=[0, 1])
test_true_negative_20, test_false_positive_20, test_false_negative_20, test_true_positive_20 = test_matrix_20.ravel()
test_specificity_20 = test_true_negative_20 / max(
    test_true_negative_20 + test_false_positive_20,
    1,
)
final_test_metrics_20 = {
    "accuracy": accuracy_score(y_test_20, test_predictions_20),
    "balanced_accuracy": balanced_accuracy_score(y_test_20, test_predictions_20),
    "malignant_recall": recall_score(y_test_20, test_predictions_20, zero_division=0),
    "precision": precision_score(y_test_20, test_predictions_20, zero_division=0),
    "specificity": test_specificity_20,
    "roc_auc": roc_auc_score(y_test_20, test_probabilities_20),
}
final_test_subgroups_20 = subgroup_metrics_20(
    y_test_20,
    test_probabilities_20,
    groups_test_20,
    threshold=selected_threshold_20,
)

print("\nFinale Test-Konfusionsmatrix:\n", test_matrix_20)
print("Finale Testmetriken:")
print(pd.Series(final_test_metrics_20).round(4))
print("\nFinale Test-Teilgruppen:")
print(final_test_subgroups_20.round(4).to_string(index=False))

selected_model_row_20 = model_comparison_20.loc[
    model_comparison_20["model"] == selected_model_name_20
].iloc[0]
project_report_20 = f'''
# Abschlussbericht: Modul 20

## Problem und Zweck
{project_definition_20['problem']} Positive Klasse: {project_definition_20['positive_class']}.
Das Modell ist ausschließlich eine Lehrdemonstration und nicht für klinische Entscheidungen vorgesehen.

## Daten und Aufteilung
Verwendet wurden {len(project_data_20)} Zeilen und {len(feature_columns_20)} numerische Merkmale.
Identifier und die nachträgliche Diagnosespalte wurden als ungeeignet beziehungsweise Leakage ausgeschlossen.
Die Aufteilung erfolgte einmalig, geschichtet und reproduzierbar in Training, Validierung und Test.

## Vergleich und Auswahl
Verglichen wurden Dummy-Baseline, logistische Regression, Entscheidungsbaum und Random Forest auf identischen Splits.
Ausgewählt wurde {selected_model_name_20} nach Validierungs-Balanced-Accuracy, Recall-Tiebreak und Artefaktgröße.
Die validierungsbasierte finale Entscheidungsschwelle beträgt {selected_threshold_20:.2f}.

## Finale Testleistung
Balanced Accuracy: {final_test_metrics_20['balanced_accuracy']:.3f}
Maligner Recall: {final_test_metrics_20['malignant_recall']:.3f}
Spezifität: {final_test_metrics_20['specificity']:.3f}
Präzision: {final_test_metrics_20['precision']:.3f}
ROC-AUC: {final_test_metrics_20['roc_auc']:.3f}

## Ressourcen
Validierungs-Fitzeit im Vergleichslauf: {selected_model_row_20['fit_seconds']:.5f} Sekunden.
Serialisierte Modellgröße im Vergleichslauf: {selected_model_row_20['artifact_kilobytes']:.2f} KB.
Diese Werte gelten nur für die aktuelle Software- und Hardwareumgebung.

## Fehleranalyse und Verantwortung
Falsch-negative Validierungsfälle wurden separat untersucht. Teilgruppenmetriken und ein synthetischer Shift-Test wurden dokumentiert.
Der Datensatz ist klein, die Standortgruppe synthetisch, und es fehlen externe Validierung, Kalibrierungsstudie und reale Überwachung.

## Nächste Schritte
Externe und zeitlich getrennte Validierung, Datenqualitätsmonitoring, Wahrscheinlichkeitskalibrierung, fachliche Prüfung der Fehlerrisiken und ein kontrollierter menschlicher Entscheidungsprozess.
'''.strip()
print("\n" + project_report_20)

### Reflexion zu Aufgabe 5

Schwellenwertanpassung verändert den Kompromiss zwischen Sensitivität und Spezifität, ohne das zugrunde liegende Ranking des Modells neu zu lernen. Die Schwelle muss auf Validierungsdaten und anhand fachlicher Kosten gewählt werden. Der Testwert darf nicht wiederholt zur Nachbesserung dienen. Auch ein guter interner Testwert reicht nicht für einen realen Einsatz. Externe Populationen, Messgeräte, zeitliche Drift, Kalibrierung, Sicherheit, Datenschutz und menschliche Aufsicht müssen separat geprüft werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.